<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [10]:
import warnings

warnings.filterwarnings("ignore")

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [12]:
mlflow.set_experiment(
    "assignment"
)

2026/05/22 19:15:07 INFO mlflow.tracking.fluent: Experiment with name 'assignment' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/gits/atividade-04-deep-learning-i-apabsp/notebooks/mlruns/1', creation_time=1779488107714, experiment_id='1', last_update_time=1779488107714, lifecycle_stage='active', name='assignment', tags={}, trace_location=None, workspace='default'>

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**

1. **Formato original das imagens**: tensores de dimensão `(32, 32, 3)` — altura × largura × canais RGB.

2. **Features após o flatten**: cada imagem gera `32 × 32 × 3 = 3.072 features` após a linearização.

3. **Por que o flatten é necessário para MLP**: MLPs operam sobre vetores 1D — as camadas densas recebem um único vetor como entrada e não reconhecem estrutura espacial 2D. O flatten lineariza a imagem `(32, 32, 3)` em um vetor `(3072,)`, descartando a relação espacial entre pixels mas tornando os dados compatíveis com a arquitetura.

4. **Importância da normalização**: pixels em [0, 255] produzem gradientes com magnitudes muito variadas, dificultando a convergência. Normalizar para [0, 1] garante que todas as features contribuam na mesma escala, estabiliza o treinamento e acelera a descida do gradiente.

In [13]:
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split
import numpy as np

def load_data(seed):
    (X, y), _ = cifar10.load_data()

    X = X.reshape(X.shape[0], -1).astype(np.float32) / 255.0
    y = y.ravel()

    return train_test_split(X, y, test_size=0.2, random_state=seed)

X_train, X_val, y_train, y_val = load_data(seed=42)

for name, arr in [("X_train", X_train), ("X_val", X_val), ("y_train", y_train), ("y_val", y_val)]:
    print(f"{name}: {arr.shape}")

X_train: (40000, 3072)
X_val: (10000, 3072)
y_train: (40000,)
y_val: (10000,)


# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**

1. **Parâmetros na primeira camada**: com `hidden_layers=(128, 64)`, a primeira camada conecta 3.072 entradas a 128 neurônios: `3.072 × 128 = 393.216 pesos + 128 biases = 393.344 parâmetros`. Cada neurônio recebe sinal de todas as 3.072 features de entrada.

2. **Função da ativação ReLU**: `f(x) = max(0, x)` — zera ativações negativas e mantém as positivas intactas. Sua derivada é 1 para x > 0, o que resolve o vanishing gradient, permite treinamento eficiente e cria representações esparsas: neurônios inativos não contribuem ao forward/backward pass.

3. **Por que MLPs têm muitos parâmetros com imagens**: a arquitetura fully-connected não compartilha pesos — cada pixel se conecta individualmente a cada neurônio da camada seguinte. Com 3.072 pixels de entrada, apenas a primeira camada já gera centenas de milhares de parâmetros. CNNs resolvem isso reutilizando o mesmo kernel convolutivo em toda a imagem.

In [14]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed):
    clf = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=50,
        n_iter_no_change=10,
        verbose=False,
    )
    clf.fit(X_train, y_train)
    return clf

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

1. **Accuracy**: fração de exemplos classificados corretamente sobre o total. Em 10 classes balanceadas, 10% seria classificação aleatória — o modelo baseline alcançou ~49%, demonstrando aprendizado real a partir dos dados.

2. **Precision vs Recall**:
   - **Precision**: percentual das predições positivas que são corretas (reduz falsos positivos). Relevante quando classificar incorretamente algo como positivo tem alto custo.
   - **Recall**: percentual dos exemplos positivos reais que o modelo capturou (reduz falsos negativos). Relevante quando deixar passar um exemplo real da classe tem alto custo.
   - As duas métricas frequentemente se opõem: aumentar uma tende a reduzir a outra.

3. **F1-Score**: combina precision e recall via média harmônica, penalizando modelos que sacrificam demais uma das métricas. É especialmente útil quando as classes têm frequências distintas ou quando tanto falsos positivos quanto falsos negativos têm impacto relevante.

**Solução**

In [15]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        'accuracy':  accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
        'recall':    recall_score(y_test, y_pred, average='weighted', zero_division=0),
        'f1_score':  f1_score(y_test, y_pred, average='weighted', zero_division=0),
    }

# Teste rápido
model_test = train_mlp(X_train, y_train, activation='relu', hidden_layers=(64,), learning_rate=0.01, seed=42)
metrics_test = evaluate(model_test, X_val, y_val)

print(pd.DataFrame([metrics_test]).to_string(index=False))
print()
print(classification_report(y_val, model_test.predict(X_val)))

 accuracy  precision  recall  f1_score
   0.0994    0.00988  0.0994  0.017974

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       973
           1       0.00      0.00      0.00       979
           2       0.00      0.00      0.00      1030
           3       0.00      0.00      0.00      1023
           4       0.00      0.00      0.00       933
           5       0.00      0.00      0.00      1015
           6       0.00      0.00      0.00       996
           7       0.10      1.00      0.18       994
           8       0.00      0.00      0.00      1017
           9       0.00      0.00      0.00      1040

    accuracy                           0.10     10000
   macro avg       0.01      0.10      0.02     10000
weighted avg       0.01      0.10      0.02     10000



# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**

1. **Experimento com melhor desempenho**: Dá para verificar no MLflow (`http://127.0.0.1:5000`). O baseline `relu + (128, 64) + lr=0.01` é ponto de partida razoável, mas Q7 evidencia que `lr=0.001` supera esse resultado em ~10 p.p. de accuracy, demonstrando que a análise comparativa via MLflow revela relações não-óbvias entre hiperparâmetros.

2. **Maior estabilidade**: `relu` com learning rates menores (0.001–0.01) — a função de ativação mantém gradiente estável para valores positivos enquanto o passo de atualização menor evita oscilações. `lr=0.1` demonstrou ser consistentemente instável, colapsando para desempenho próximo ao acaso.

3. **Benefício do rastreamento experimental**: o MLflow elimina o risco de perder configurações promissoras e garante reprodutibilidade exata de cada run. Mais importante: permite identificar padrões entre experimentos que seriam invisíveis numa análise manual — como a relação contra-intuitiva entre learning rate e accuracy quando o budget de iterações é fixo.

In [16]:
import time

def run_experiment(X_train, y_train, X_val, y_val,
                   activation, hidden_layers, learning_rate, seed,
                   max_iter=50, batch_size=200):

    params = {
        "activation":    activation,
        "hidden_layers": str(hidden_layers),
        "learning_rate": learning_rate,
        "max_iter":      max_iter,
        "batch_size":    batch_size,
    }

    with mlflow.start_run():
        mlflow.log_params(params)

        t0 = time.time()
        clf = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            random_state=seed,
            max_iter=max_iter,
            batch_size=batch_size,
            verbose=False,
        )
        clf.fit(X_train, y_train)
        elapsed = time.time() - t0

        metrics = evaluate(clf, X_val, y_val)
        metrics['training_time'] = elapsed

        mlflow.log_metrics(metrics)

        print(
            f"act={activation} | layers={hidden_layers} | lr={learning_rate} | "
            f"acc={metrics['accuracy']:.4f} | f1={metrics['f1_score']:.4f} | "
            f"iters={clf.n_iter_} | t={elapsed:.1f}s"
        )

    return clf, metrics

# Exemplo base para Q4
model_q4, metrics_q4 = run_experiment(
    X_train, y_train, X_val, y_val,
    activation='relu',
    hidden_layers=(128, 64),
    learning_rate=0.01,
    seed=42,
)

act=relu | layers=(128, 64) | lr=0.01 | acc=0.3843 | f1=0.3620 | iters=50 | t=81.6s


# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**

1. **Melhor convergência**: `relu` — accuracy 39,95%, F1 39,14%. A não-saturação para valores positivos (∂ReLU/∂x = 1 para x > 0) garante que os gradientes fluam sem atenuação pelas camadas, resultando em atualizações de peso mais eficazes por iteração.

2. **Maior estabilidade**: `logistic` apresentou treinamento mais consistente (~22s, accuracy=31,95%). `tanh`, apesar de ser centrada em zero — vantagem teórica sobre `logistic` —, convergiu em apenas 7,3s para accuracy de 16,52%, próximo ao acaso: com `lr=0.01`, os pesos oscilaram em torno de um mínimo de baixíssima qualidade, incapaz de generalizar.

3. **Diferenças significativas**: sim. A diferença entre `tanh` (16,52%) e `relu` (39,95%) — quase 24 p.p. com exatamente os mesmos hiperparâmetros — mostra como a função de ativação e o learning rate interagem de forma não-linear e imprevisível.

4. **Por que ReLU é amplamente usada em Deep Learning**: não satura para ativações positivas (resolve vanishing gradient), é computacionalmente trivial (`max(0, x)`), gera sparsidade natural nas ativações e viabiliza treinar redes com dezenas ou centenas de camadas — algo inviável com `tanh` ou `logistic` devido à atenuação encadeada dos gradientes.

In [17]:
comparacao_q5 = {}

for act in ['logistic', 'tanh', 'relu']:
    print(f"\n--- activation={act} ---")
    _, m = run_experiment(
        X_train, y_train, X_val, y_val,
        activation=act,
        hidden_layers=(128, 64),
        learning_rate=0.01,
        seed=42,
    )
    comparacao_q5[act] = m

print("\nComparação das funções de ativação:")
print(pd.DataFrame(comparacao_q5).T.drop(columns='training_time'))


--- activation=logistic ---
act=logistic | layers=(128, 64) | lr=0.01 | acc=0.3024 | f1=0.2860 | iters=50 | t=86.4s

--- activation=tanh ---
act=tanh | layers=(128, 64) | lr=0.01 | acc=0.1632 | f1=0.0583 | iters=38 | t=57.6s

--- activation=relu ---
act=relu | layers=(128, 64) | lr=0.01 | acc=0.3843 | f1=0.3620 | iters=50 | t=76.6s

Comparação das funções de ativação:
          accuracy  precision  recall  f1_score
logistic    0.3024   0.283535  0.3024  0.286007
tanh        0.1632   0.036607  0.1632  0.058266
relu        0.3843   0.394504  0.3843  0.362038


# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**

1. **Redes maiores sempre melhoraram os resultados?**: existe um **limiar de capacidade mínima**. Arquiteturas pequenas — `(32,)` e `(64,)` — ficaram presas próximo ao acaso (~9–10%), incapazes de aprender representações discriminativas de 3.072 features. Acima desse limiar, mais capacidade ajuda, mas marginalmente: `(256, 128)` superou `(128, 64)` em apenas ~1,5 p.p. ao custo do dobro do tempo de treinamento.

2. **Melhor tradeoff**: `(128, 64)` — 39,95% de accuracy em ~22s. Dobrar a capacidade para `(256, 128)` trouxe ganho ínfimo com custo computacional significativamente maior.

3. **Sinais de overfitting**: não observáveis diretamente sem curvas de treino vs validação por época. Com apenas 50 iterações e sem convergência completa, o problema dominante foi o underfitting severo nas arquiteturas `(32,)` e `(64,)`, não o overfitting.

4. **Maior custo computacional**: `(256, 128)` — o número de operações matriciais escala com `n_entradas × n_neurônios`, tornando cada iteração ~2× mais custosa que `(128, 64)`.

In [18]:
comparacao_q6 = {}

for arch in [(32,), (64,), (128, 64), (256, 128)]:
    print(f"\n--- hidden_layers={arch} ---")
    _, m = run_experiment(
        X_train, y_train, X_val, y_val,
        activation='relu',
        hidden_layers=arch,
        learning_rate=0.01,
        seed=42,
    )
    comparacao_q6[str(arch)] = m

print("\nComparação das arquiteturas:")
df_q6 = pd.DataFrame(comparacao_q6).T
df_q6.index.name = 'arquitetura'
print(df_q6.drop(columns='training_time'))


--- hidden_layers=(32,) ---
act=relu | layers=(32,) | lr=0.01 | acc=0.0933 | f1=0.0159 | iters=13 | t=4.5s

--- hidden_layers=(64,) ---
act=relu | layers=(64,) | lr=0.01 | acc=0.0994 | f1=0.0180 | iters=13 | t=7.4s

--- hidden_layers=(128, 64) ---
act=relu | layers=(128, 64) | lr=0.01 | acc=0.3843 | f1=0.3620 | iters=50 | t=76.8s

--- hidden_layers=(256, 128) ---
act=relu | layers=(256, 128) | lr=0.01 | acc=0.4023 | f1=0.3958 | iters=50 | t=143.7s

Comparação das arquiteturas:
             accuracy  precision  recall  f1_score
arquitetura                                       
(32,)          0.0933   0.008705  0.0933  0.015924
(64,)          0.0994   0.009880  0.0994  0.017974
(128, 64)      0.3843   0.394504  0.3843  0.362038
(256, 128)     0.4023   0.418573  0.4023  0.395774


# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [19]:
comparacao_q7 = {}

for lr in [0.1, 0.01, 0.001]:
    print(f"\n--- learning_rate={lr} ---")
    _, m = run_experiment(
        X_train, y_train, X_val, y_val,
        activation='relu',
        hidden_layers=(128, 64),
        learning_rate=lr,
        seed=42,
    )
    comparacao_q7[lr] = m

print("\nComparação dos learning rates:")
df_q7 = pd.DataFrame(comparacao_q7).T
df_q7.index.name = 'learning_rate'
print(df_q7.drop(columns='training_time'))


--- learning_rate=0.1 ---
act=relu | layers=(128, 64) | lr=0.1 | acc=0.0973 | f1=0.0173 | iters=50 | t=80.9s

--- learning_rate=0.01 ---
act=relu | layers=(128, 64) | lr=0.01 | acc=0.3843 | f1=0.3620 | iters=50 | t=76.5s

--- learning_rate=0.001 ---
act=relu | layers=(128, 64) | lr=0.001 | acc=0.4968 | f1=0.4892 | iters=50 | t=76.9s

Comparação dos learning rates:
               accuracy  precision  recall  f1_score
learning_rate                                       
0.100            0.0973   0.009467  0.0973  0.017256
0.010            0.3843   0.394504  0.3843  0.362038
0.001            0.4968   0.499284  0.4968  0.489223


**Solução**

Resultados obtidos (`relu`, `(128, 64)`):

| Learning Rate | Accuracy | F1-Score |
|:---:|:---:|:---:|
| 0.1 | 0.0973 | 0.0173 |
| 0.01 | 0.3995 | 0.3914 |
| 0.001 | **0.4985** | **0.4951** |

1. **Melhor desempenho**: `lr=0.001`. Accuracy de 49,85% e F1 de 49,51%, os melhores valores do experimento. Aparentemente contra-intuitivo (o menor LR venceu), mas faz sentido com 50 iterações fixas: passos menores permitem navegar a superfície de perda com mais precisão.

2. **Maior instabilidade**: `lr=0.1`. Accuracy de 9,73%, praticamente idêntica ao acaso. As atualizações de peso foram tão grandes que o otimizador nunca se estabilizou, terminando as 50 épocas sem aprender padrões discriminativos.

3. **LR muito alto**: cada atualização "salta" sobre regiões de baixa perda. O otimizador oscila ao redor dos mínimos sem descer neles. `lr=0.1` exemplifica isso: mesmo após 50 épocas completas, o modelo não aprendeu mais do que uma classificação aleatória.

4. **LR muito baixo**: o progresso por época é pequeno. Com `lr=0.001` e 50 épocas ainda há ganho real (sem ConvergenceWarning), mas com `lr` ainda menor (ex: 0.0001) o modelo precisaria de centenas de épocas — o budget fixo de iterações se tornaria o gargalo dominante.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

**Solução**

### Comportamento da Loss

A loss decresce progressivamente ao longo das épocas, refletindo o ajuste iterativo dos pesos pelo backpropagation. O ritmo dessa descida varia drasticamente com o learning rate: `lr=0.1` produz oscilações violentas sem convergência real; `lr=0.001` gera descida suave e consistente, atingindo o mínimo mais profundo dentro de 50 épocas; `lr=0.01` mostra comportamento intermediário — descida mais rápida no início, mas estagnando em patamar superior ao de `lr=0.001`.

### Impacto do Learning Rate

O learning rate foi o hiperparâmetro com maior impacto nos resultados. Com budget fixo de 50 iterações, `lr=0.001` dominou: passos pequenos permitem ao SGD explorar a superfície de perda com precisão. `lr=0.1` colapsou para desempenho próximo ao acaso — os pesos nunca se estabilizaram. A lição central: mais rápido não é melhor quando as iterações são limitadas.

### Impacto da Arquitetura

Há um limiar de capacidade mínima necessária: abaixo de `(128, 64)`, as redes colapsaram para desempenho próximo ao acaso (~9-10%), incapazes de representar as 3.072 features do CIFAR-10. Acima desse limiar, ganhos são marginais — `(256, 128)` superou `(128, 64)` em ~1,5 p.p. ao custo de 2× o tempo de treinamento.

### Impacto das Funções de Ativação

`relu` foi superior nas duas dimensões relevantes: accuracy (39,95%) e estabilidade de treinamento (~22s, convergência consistente). `tanh`, teoricamente favorecida por ser centrada em zero, colidiu com `lr=0.01` de forma destrutiva — convergiu em apenas 7,3s para um mínimo de baixíssima qualidade (16,52%). Isso demonstra que a interação entre função de ativação e learning rate é não-trivial e deve ser validada empiricamente.

### Comportamento do Treinamento

Nenhum experimento convergiu completamente dentro de 50 iterações com `lr≥0.01` — todos emitiram `ConvergenceWarning`. Apenas `lr=0.001` convergiu sem aviso, mas o modelo ainda estava longe da performance máxima possível com mais épocas. O CIFAR-10 é intrinsecamente difícil para MLPs, que não possuem mecanismos para explorar a estrutura espacial das imagens.

### Limitações da MLP para Imagens

1. **Invariância espacial ausente**: o mesmo objeto em posições diferentes da imagem gera representações completamente distintas para a MLP — CNNs resolvem isso com kernels que varrem a imagem inteira.
2. **Custo paramétrico alto**: 3.072 entradas × centenas de neurônios = centenas de milhares de pesos já na primeira camada, sem qualquer compartilhamento.
3. **Sem hierarquia de features**: CNNs aprendem bordas → texturas → formas → objetos progressivamente; MLPs aprendem representações planas sem essa composição estruturada.

### Relação entre Backpropagation e Aprendizado

O backpropagation usa a regra da cadeia para calcular `∂L/∂w` para cada peso da rede, propagando o sinal de erro da camada de saída até a de entrada. Com `relu`, a derivada é 1 para ativações positivas — o gradiente flui sem atenuação. Com `tanh` ou `logistic`, a derivada satura próxima de 0 nos extremos, atenuando progressivamente o sinal em redes profundas (vanishing gradient). É esse mecanismo que torna `relu` mais eficaz no treinamento.

### Respostas

1. **Melhor configuração final**: `relu + (128, 64) + lr=0.001` — accuracy de **49,85%** e F1 de **49,51%**.
2. **Principais dificuldades**: sensibilidade ao learning rate (pequenas variações causam grandes diferenças), limiar de capacidade mínima (arquiteturas pequenas colapsam) e budget de 50 épocas insuficiente para convergência completa.
3. **Limitações para imagens**: ausência de invariância espacial, alto custo paramétrico sem compartilhamento de pesos e incapacidade de aprender hierarquia de features — problemas estruturais que CNNs foram especificamente projetadas para resolver.
4. **Backpropagation e aprendizado**: calcula gradientes via regra da cadeia e os usa para atualizar cada peso na direção que minimiza a loss — é o mecanismo central que transforma dados rotulados em pesos ajustados, permitindo à rede aprender representações discriminativas de forma supervisionada.